In [ ]:
# In[1]:

import pandas as pd

# Load files
df_metric = pd.read_csv('metric.csv')
df_trace = pd.read_csv('trace.csv')
df_log = pd.read_csv('log.csv')
df_error = pd.read_csv('error_logs.csv')

# Helper to produce meta summary (row count, min/max timestamp as Unix seconds) and parsed datetime column
def meta_summary_df(df, ts_col='timestamp'):
    row_count = len(df)
    if row_count == 0:
        min_ts = None
        max_ts = None
    else:
        # ensure numeric timestamps
        df[ts_col] = pd.to_numeric(df[ts_col], errors='coerce').astype('Int64')
        min_ts = int(df[ts_col].min())
        max_ts = int(df[ts_col].max())
        # add parsed datetime (UTC) column for potential downstream use
        df['_ts_dt_utc'] = pd.to_datetime(df[ts_col], unit='s', utc=True)
    meta = pd.DataFrame([{'row_count': row_count, 'min_ts': min_ts, 'max_ts': max_ts}])
    return meta

# Metric summaries
metric_meta = meta_summary_df(df_metric)
metric_cmdbs = pd.DataFrame({'cmdb_id': pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()))}).head(50)
metric_kpi_unique = pd.DataFrame({'kpi_name': pd.Series(sorted(df_metric['kpi_name'].dropna().unique()))})
metric_kpi_top20 = df_metric['kpi_name'].value_counts().rename_axis('kpi_name').reset_index(name='count').head(20)

# Trace summaries
trace_meta = meta_summary_df(df_trace)
trace_cmdbs = pd.DataFrame({'cmdb_id': pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()))}).head(50)
trace_name_unique = pd.DataFrame({'trace_name': pd.Series(sorted(df_trace['trace_name'].dropna().unique()))})
trace_name_top20 = df_trace['trace_name'].value_counts().rename_axis('trace_name').reset_index(name='count').head(20)

# Log summaries
log_meta = meta_summary_df(df_log)
log_cmdbs = pd.DataFrame({'cmdb_id': pd.Series(sorted(df_log['cmdb_id'].dropna().unique()))}).head(50)
log_name_unique = pd.DataFrame({'log_name': pd.Series(sorted(df_log['log_name'].dropna().unique()))})
log_name_top20 = df_log['log_name'].value_counts().rename_axis('log_name').reset_index(name='count').head(20)

# Error logs summaries
error_meta = meta_summary_df(df_error)
error_cmdb_top20 = df_error['cmdb_id'].value_counts().rename_axis('cmdb_id').reset_index(name='count').head(20)
# top 10 distinct messages (or first 10 rows of message if many duplicates)
error_messages_top10 = pd.DataFrame({'message': df_error['message'].drop_duplicates().head(10).reset_index(drop=True)})

# Compact display: show metadata and limited lists as requested
metric_meta, metric_cmdbs, metric_kpi_unique.head(100), metric_kpi_top20, \
trace_meta, trace_cmdbs, trace_name_unique.head(100), trace_name_top20, \
log_meta, log_cmdbs, log_name_unique.head(100), log_name_top20, \
error_meta, error_messages_top10, error_cmdb_top20

```
Out[1]:
```
```python
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: 1900 rows; timestamp range 1705757820 to 1705759260 (Unix seconds). "
    "cmdb_id examples: carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db. "
    "KPIs present: cpu, mem, socket, diskio, latency-50, latency-90, workload, error. "
    "Top KPI counts: cpu 375, mem 375, socket 375, diskio 200, latency-90 175, latency-50 175, workload 175, error 50.\n"
    "- trace.csv: 0 rows (no trace data available).\n"
    "- log.csv: 460 rows; timestamp range 1705757820 to 1705759260 (Unix seconds). "
    "cmdb_id examples: carts, carts-db, catalogue, front-end, orders, orders-db, payment, queue-master, session-db, shipping, user, user-db. "
    "Log names: log.error_count and log.row_count (each with 230 occurrences).\n"
    "- error_logs.csv: 0 rows (no error log messages available)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   row_count      min_ts      max_ts
0       1900  1705757820  1705759260,               cmdb_id
0               carts
1            carts-db
2           catalogue
3        catalogue-db
4           front-end
5              orders
6           orders-db
7             payment
8        queue-master
9            rabbitmq
10  rabbitmq-exporter
11         session-db
12           shipping
13               user
14            user-db,      kpi_name
0         cpu
1      diskio
2       error
3  latency-50
4  latency-90
5         mem
6      socket
7    workload,      kpi_name  count
0         cpu    375
1         mem    375
2      socket    375
3      diskio    200
4  latency-90    175
5  latency-50    175
6    workload    175
7       error     50,    row_count min_ts max_ts
0          0   None   None, Empty DataFrame
Columns: [cmdb_id]
Index: [], Empty DataFrame
Columns: [trace_name]
Index: [], Empty DataFrame
Columns: [trace_name, count]
Index: [],    row_count      min_ts      max_ts
0        460  1705757820  1705759260,          cmdb_id
0          carts
1       carts-db
2      catalogue
3      front-end
4         orders
5      orders-db
6        payment
7   queue-master
8     session-db
9       shipping
10          user
11       user-db,           log_name
0  log.error_count
1    log.row_count,           log_name  count
0  log.error_count    230
1    log.row_count    230,    row_count min_ts max_ts
0          0   None   None, Empty DataFrame
Columns: [message]
Index: [], Empty DataFrame
Columns: [cmdb_id, count]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse previously loaded dataframes: df_metric, df_trace, df_log, df_error
# Ensure timestamp numeric and parsed as UTC datetimes
df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
df_metric['_ts_dt_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# 1) Global thresholds (P95 and P5) per component-KPI using entire metric series
grouped_values = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
p95 = grouped_values.quantile(0.95).rename('p95')
p5 = grouped_values.quantile(0.05).rename('p5')
thresholds = pd.concat([p95, p5], axis=1).reset_index()

# Count of distinct groups processed
total_groups = thresholds.shape[0]

# 2) Parse incident window (UTC) and filter metric rows within the window (inclusive)
start_ts = pd.to_datetime('2024-01-20 13:34:02', utc=True)
end_ts = pd.to_datetime('2024-01-20 14:04:02', utc=True)
df_win = df_metric[(df_metric['_ts_dt_utc'] >= start_ts) & (df_metric['_ts_dt_utc'] <= end_ts)].copy()

# 3) Merge thresholds into window rows so we can compare per-row
df_win = df_win.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Mark anomalies within window relative to global thresholds
df_win['anomaly_high'] = (df_win['value'] >= df_win['p95']).astype(int)
df_win['anomaly_low'] = (df_win['value'] <= df_win['p5']).astype(int)

# Aggregations per group within the incident window
grouped_win = df_win.groupby(['cmdb_id', 'kpi_name'])

anomaly_high_count = grouped_win['anomaly_high'].sum().rename('anomaly_high_count')
anomaly_low_count = grouped_win['anomaly_low'].sum().rename('anomaly_low_count')
max_value_in_window = grouped_win['value'].max().rename('max_value_in_window')
min_value_in_window = grouped_win['value'].min().rename('min_value_in_window')

# Earliest timestamps (Unix seconds) for first high/low anomaly in the window
def earliest_ts_for_flag(df_group, flag_col):
    flagged = df_group.loc[df_group[flag_col] == 1, 'timestamp']
    if flagged.empty:
        return pd.NA
    return int(flagged.min())

earliest_high = grouped_win.apply(lambda g: earliest_ts_for_flag(g, 'anomaly_high')).rename('earliest_high_timestamp')
earliest_low = grouped_win.apply(lambda g: earliest_ts_for_flag(g, 'anomaly_low')).rename('earliest_low_timestamp')

# Combine all aggregated series into a single DataFrame
agg = pd.concat([
    anomaly_high_count,
    anomaly_low_count,
    earliest_high,
    earliest_low,
    max_value_in_window,
    min_value_in_window
], axis=1).reset_index()

# Bring in the global thresholds p95/p5
agg = agg.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# 4) Filter groups that have any anomalies within the window and keep requested columns
agg['total_anomalies'] = agg['anomaly_high_count'] + agg['anomaly_low_count']
anomalous_groups = agg[agg['total_anomalies'] > 0].copy()

# Select and order columns as requested
result_cols = [
    'cmdb_id', 'kpi_name', 'p95', 'p5',
    'anomaly_high_count', 'anomaly_low_count',
    'earliest_high_timestamp', 'earliest_low_timestamp',
    'max_value_in_window', 'min_value_in_window'
]
anomalies_top20 = anomalous_groups.sort_values(by='total_anomalies', ascending=False).head(20)[result_cols]

# Confirm trace.csv and error_logs.csv are empty
trace_empty = (df_trace.shape[0] == 0)
error_logs_empty = (df_error.shape[0] == 0)

# Return compact results: top anomaly rows, total group count, and emptiness confirmations
anomalies_top20, total_groups, trace_empty, error_logs_empty

```
Out[2]:
```
summary = (
    "Results summary:\n"
    "- Global thresholds (P95/P5) were computed for 76 distinct component-KPI groups using the full metric.csv series.\n"
    "- Incident window: 2024-01-20 13:34:02 UTC through 2024-01-20 14:04:02 UTC.\n"
    "- Anomalous groups within the window (top examples by total anomalies):\n"
    "  1) session-db / socket: p95=3.0, p5=3.0, high=25, low=25, earliest_high_timestamp=1705757820, max=3.0, min=3.0\n"
    "  2) catalogue-db / socket: p95=4.0, p5=4.0, high=25, low=25, earliest_high_timestamp=1705757820, max=4.0, min=4.0\n"
    "  3) catalogue-db / diskio: p95=0.0, p5=0.0, high=25, low=25, earliest_high_timestamp=1705757820, max=0.0, min=0.0\n"
    "  4) rabbitmq-exporter / socket: p95=1.0, p5=1.0, high=25, low=25, earliest_high_timestamp=1705757820, max=1.0, min=1.0\n"
    "  5) rabbitmq / diskio: p95=0.0, p5=0.0, high=25, low=25, earliest_high_timestamp=1705757820, max=0.0, min=0.0\n"
    "  6) queue-master / socket: p95=3.0, p5=3.0, high=25, low=25, earliest_high_timestamp=1705757820, max=3.0, min=3.0\n"
    "  7) rabbitmq-exporter / mem: p95≈4.54656e+06, p5≈4.54656e+06, high=25, low=25, earliest_high_timestamp=1705757820, max≈4.54656e+06, min≈4.54656e+06\n"
    "  8) rabbitmq / socket: p95=11.0, p5=11.0, high=25, low=25, earliest_high_timestamp=1705757820, max=11.0, min=11.0\n"
    "  9) catalogue / socket: p95=6.0, p5=6.0, high=25, low=25, earliest_high_timestamp=1705757820, max=6.0, min=6.0\n"
    " 10) front-end / error: p95=0.0, p5=0.0, high=25, low=24, earliest_high_timestamp=1705757820, max≈0.03345, min=0.0\n"
    "- Several other groups show mixed high/low anomalies; the returned table included up to 20 top groups (sorted by total anomalies).\n"
    "- Total distinct component-KPI groups processed: 76.\n"
    "- trace.csv empty: True. error_logs.csv empty: True.\n\n"
    "Interpretation: multiple components (notably messaging-related and some catalogue/session components) exhibit many values at or beyond their global P95 and P5 within the incident window. The output table (top 20) should be consulted for exact per-group counts and timestamps."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name           p95            p5  anomaly_high_count  anomaly_low_count  earliest_high_timestamp  earliest_low_timestamp  max_value_in_window  min_value_in_window
59         session-db   socket  3.000000e+00  3.000000e+00                  25                 25               1705757820              1705757820         3.000000e+00         3.000000e+00
19       catalogue-db   socket  4.000000e+00  4.000000e+00                  25                 25               1705757820              1705757820         4.000000e+00         4.000000e+00
17       catalogue-db   diskio  0.000000e+00  0.000000e+00                  25                 25               1705757820              1705757820         0.000000e+00         0.000000e+00
55  rabbitmq-exporter   socket  1.000000e+00  1.000000e+00                  25                 25               1705757820              1705757820         1.000000e+00         1.000000e+00
50           rabbitmq   diskio  0.000000e+00  0.000000e+00                  25                 25               1705757820              1705757820         0.000000e+00         0.000000e+00
48       queue-master   socket  3.000000e+00  3.000000e+00                  25                 25               1705757820              1705757820         3.000000e+00         3.000000e+00
54  rabbitmq-exporter      mem  4.546560e+06  4.546560e+06                  25                 25               1705757820              1705757820         4.546560e+06         4.546560e+06
52           rabbitmq   socket  1.100000e+01  1.100000e+01                  25                 25               1705757820              1705757820         1.100000e+01         1.100000e+01
14          catalogue   socket  6.000000e+00  6.000000e+00                  25                 25               1705757820              1705757820         6.000000e+00         6.000000e+00
21          front-end    error  0.000000e+00  0.000000e+00                  25                 24               1705757820              1705757820         3.345000e-02         0.000000e+00
28             orders    error  0.000000e+00  0.000000e+00                  25                 24               1705757820              1705757820         3.325000e-02         0.000000e+00
39            payment   diskio  0.000000e+00  0.000000e+00                  25                 24               1705757820              1705757880         1.476333e+05         0.000000e+00
37          orders-db   socket  5.000000e+00  4.746667e+00                  23                  2               1705757820              1705758900         5.000000e+00         4.216667e+00
43            payment   socket  1.000000e+01  7.000000e+00                  12                 12               1705758600              1705757820         1.000000e+01         7.000000e+00
9            carts-db   socket  7.000000e+00  6.000000e+00                   8                 16               1705758840              1705757820         7.000000e+00         6.000000e+00
75            user-db   socket  9.000000e+00  8.000000e+00                   5                 18               1705757820              1705758060         9.000000e+00         8.000000e+00
57         session-db   diskio  2.039242e+04  0.000000e+00                   2                 20               1705758120              1705757880         2.123244e+04         0.000000e+00
18       catalogue-db      mem  4.307560e+08  4.307354e+08                   2                  6               1705757880              1705758060         4.307681e+08         4.307354e+08
70               user   socket  1.784333e+01  1.600000e+01                   2                  5               1705759200              1705758060         1.800000e+01         1.600000e+01
25          front-end   socket  9.176667e+00  7.000000e+00                   2                  5               1705758900              1705757880         9.633333e+00         7.000000e+00, 76, True, True)```
```

In [ ]:
# In[3]:

# Reuse existing variables: df_metric, thresholds, df_win
import pandas as pd

# Ensure thresholds exist; if not, compute (global P95/P5 per group)
if 'thresholds' not in globals():
    grouped_values = df_metric.groupby(['cmdb_id', 'kpi_name'])['value']
    p95 = grouped_values.quantile(0.95).rename('p95')
    p5 = grouped_values.quantile(0.05).rename('p5')
    thresholds = pd.concat([p95, p5], axis=1).reset_index()

# Incident window (already used before) - redefine to be explicit and ensure UTC
start_ts = pd.to_datetime('2024-01-20 13:34:02', utc=True)
end_ts = pd.to_datetime('2024-01-20 14:04:02', utc=True)

# Ensure df_metric has parsed UTC timestamps column
if '_ts_dt_utc' not in df_metric.columns:
    df_metric['timestamp'] = pd.to_numeric(df_metric['timestamp'], errors='coerce').astype('Int64')
    df_metric['_ts_dt_utc'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Filter metric rows to the incident window
df_win = df_metric[(df_metric['_ts_dt_utc'] >= start_ts) & (df_metric['_ts_dt_utc'] <= end_ts)].copy()

# Candidate services to restrict analysis
candidates = ['carts','catalogue','orders','payment','user']

# Merge global thresholds into the windowed rows
df_win = df_win.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

# Restrict to candidate services
df_win_cand = df_win[df_win['cmdb_id'].isin(candidates)].copy()

# If there are no rows for candidates, prepare empty outputs
if df_win_cand.empty:
    anomalies_top20 = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','p95','p5','anomaly_high_count','anomaly_low_count',
        'earliest_high_timestamp','earliest_low_timestamp','max_value_in_window','min_value_in_window'
    ])
    zero_anomaly_services = pd.DataFrame({'cmdb_id': candidates, 'total_anomalies': 0})
else:
    # Mark anomalies relative to pre-computed global thresholds
    df_win_cand['anomaly_high'] = (df_win_cand['value'] >= df_win_cand['p95']).astype(int)
    df_win_cand['anomaly_low'] = (df_win_cand['value'] <= df_win_cand['p5']).astype(int)

    # Group and aggregate per (cmdb_id, kpi_name)
    g = df_win_cand.groupby(['cmdb_id', 'kpi_name'])

    anomaly_high_count = g['anomaly_high'].sum().rename('anomaly_high_count')
    anomaly_low_count = g['anomaly_low'].sum().rename('anomaly_low_count')
    max_value_in_window = g['value'].max().rename('max_value_in_window')
    min_value_in_window = g['value'].min().rename('min_value_in_window')

    # Earliest timestamps for high/low anomalies (Unix seconds)
    def earliest_flag_ts(df_group, flag_col):
        flagged = df_group.loc[df_group[flag_col] == 1, 'timestamp']
        if flagged.empty:
            return pd.NA
        return int(flagged.min())

    earliest_high = g.apply(lambda grp: earliest_flag_ts(grp, 'anomaly_high')).rename('earliest_high_timestamp')
    earliest_low = g.apply(lambda grp: earliest_flag_ts(grp, 'anomaly_low')).rename('earliest_low_timestamp')

    agg = pd.concat([
        anomaly_high_count, anomaly_low_count,
        earliest_high, earliest_low,
        max_value_in_window, min_value_in_window
    ], axis=1).reset_index()

    # Merge global thresholds (p95/p5) for each group
    agg = agg.merge(thresholds, on=['cmdb_id', 'kpi_name'], how='left')

    # Compute total anomalies and filter groups with any anomalies (>0)
    agg['total_anomalies'] = agg['anomaly_high_count'] + agg['anomaly_low_count']
    anomalous_groups = agg[agg['total_anomalies'] > 0].copy()

    # Select requested columns and sort by total anomalies desc, limit 20 rows
    result_cols = [
        'cmdb_id','kpi_name','p95','p5',
        'anomaly_high_count','anomaly_low_count',
        'earliest_high_timestamp','earliest_low_timestamp',
        'max_value_in_window','min_value_in_window'
    ]
    anomalies_top20 = anomalous_groups.sort_values(by='total_anomalies', ascending=False).head(20)[result_cols]

    # Determine candidate services that had zero anomalies across all their KPIs in the window
    svc_anom = agg.groupby('cmdb_id')['total_anomalies'].sum().reset_index()
    # Include candidates that had no rows at all in svc_anom (treat as zero anomalies)
    svc_df = pd.DataFrame({'cmdb_id': candidates}).merge(svc_anom, on='cmdb_id', how='left').fillna(0)
    zero_anomaly_services = svc_df[svc_df['total_anomalies'] == 0].reset_index(drop=True)

# Display compact results: anomalies table (max 20 rows) and list of zero-anomaly candidate services
anomalies_top20, zero_anomaly_services

```
Out[3]:
```
```python
summary = (
    "Summary for candidate services (window 2024-01-20 13:34:02 to 14:04:02 UTC):\n\n"
    "1) All five candidate services showed anomalies in the incident window (no candidate had zero anomalies).\n\n"
    "2) Top affected component-KPI groups (by total anomalies = high + low):\n"
    "   - catalogue / socket: p95=6.0, p5=6.0, high=25, low=25 (total 50). Earliest anomaly at 1705757820. Values constant at 6.0 in window.\n"
    "   - orders / error: p95=0.0, p5=0.0, high=25, low=24 (total 49). Earliest anomaly at 1705757820. max≈0.03325, min=0.0.\n"
    "   - payment / diskio: p95=0.0, p5=0.0, high=25, low=24 (total 49). Earliest anomalies at 1705757820 (low) and 1705757880 (low/high). max≈147633.3, min=0.0.\n"
    "   - payment / socket: p95=10.0, p5=7.0, high=12, low=12 (total 24). Earliest_high=1705758600, earliest_low=1705757820. max=10.0, min=7.0.\n"
    "   - user / socket: p95≈17.84333, p5=16.0, high=2, low=5 (total 7). Earliest_high=1705759200, earliest_low=1705758060.\n\n"
    "3) Other notable candidate groups with smaller but nonzero anomaly counts (examples):\n"
    "   - carts: socket, cpu, latency-50, latency-90, mem, workload each show anomalies (each group in window had small counts, e.g., many with 2 high + 2 low).\n"
    "   - catalogue: besides socket, several KPIs (cpu, mem, latency-50, latency-90, workload) show modest anomalies (typically 2 high + 2 low).\n"
    "   - orders: besides error, cpu, mem, latency-50, latency-90 show modest anomalies (typically 2 high + 2 low).\n\n"
    "4) Interpretation: every candidate service (carts, catalogue, orders, payment, user) exhibited metric values hitting their global P95 and/or P5 during the incident window. "
    "Catalogue.socket and orders.error and payment.diskio are the most strongly affected groups (largest anomaly counts). Earliest anomalies for many groups occurred at or near the start of the window (Unix 1705757820).\n"
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name           p95            p5  anomaly_high_count  anomaly_low_count  earliest_high_timestamp  earliest_low_timestamp  max_value_in_window  min_value_in_window
10  catalogue      socket  6.000000e+00  6.000000e+00                  25                 25               1705757820              1705757820         6.000000e+00         6.000000e+00
13     orders       error  0.000000e+00  0.000000e+00                  25                 24               1705757820              1705757820         3.325000e-02         0.000000e+00
20    payment      diskio  0.000000e+00  0.000000e+00                  25                 24               1705757820              1705757880         1.476333e+05         0.000000e+00
24    payment      socket  1.000000e+01  7.000000e+00                  12                 12               1705758600              1705757820         1.000000e+01         7.000000e+00
30       user      socket  1.784333e+01  1.600000e+01                   2                  5               1705759200              1705758060         1.800000e+01         1.600000e+01
4       carts      socket  1.194333e+01  9.083333e+00                   2                  2               1705758840              1705757940         1.216667e+01         8.783333e+00
0       carts         cpu  1.706139e+00  1.392543e+00                   2                  2               1705758840              1705757940         2.008599e+00         1.382357e+00
1       carts  latency-50  2.221415e-02  1.919985e-02                   2                  2               1705759020              1705757940         2.278981e-02         1.907074e-02
7   catalogue  latency-50  3.069059e-03  3.012182e-03                   2                  2               1705758300              1705758960         3.096369e-03         3.010929e-03
6   catalogue         cpu  1.784866e-01  1.541886e-01                   2                  2               1705757820              1705758480         1.851791e-01         1.505660e-01
9   catalogue         mem  6.311281e+06  6.275231e+06                   2                  2               1705758900              1705757880         6.312073e+06         6.273980e+06
8   catalogue  latency-90  4.724307e-03  4.621928e-03                   2                  2               1705758300              1705758960         4.773465e-03         4.619672e-03
11  catalogue    workload  4.263715e+00  3.941610e+00                   2                  2               1705758060              1705757940         4.376683e+00         3.846600e+00
5       carts    workload  8.736123e+00  7.893767e+00                   2                  2               1705758180              1705758540         9.067000e+00         7.699983e+00
2       carts  latency-90  5.034513e-02  4.123827e-02                   2                  2               1705758840              1705757880         5.871154e-02         4.096137e-02
3       carts         mem  2.118314e+08  2.113366e+08                   2                  2               1705759200              1705757940         2.118410e+08         2.113350e+08
15     orders  latency-90  2.567568e-01  4.749242e-02                   2                  2               1705758600              1705757820         2.702292e-01         4.746104e-02
14     orders  latency-50  4.448424e-02  3.745899e-02                   2                  2               1705758720              1705757820         4.473684e-02         3.730519e-02
12     orders         cpu  2.478843e+00  1.101955e+00                   2                  2               1705758120              1705758480         2.750972e+00         1.083708e+00
16     orders         mem  3.330646e+08  3.325721e+08                   2                  2               1705758060              1705757820         3.330725e+08         3.323807e+08, Empty DataFrame
Columns: [cmdb_id, total_anomalies]
Index: [])```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse existing df_log if present; otherwise load
if 'df_log' not in globals():
    df_log = pd.read_csv('log.csv')

# Ensure timestamp numeric and parsed as UTC datetime
df_log['timestamp'] = pd.to_numeric(df_log['timestamp'], errors='coerce').astype('Int64')
df_log['_ts_dt_utc'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)

# Ensure value numeric
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Incident window (UTC)
start_ts = pd.to_datetime('2024-01-20 13:34:02', utc=True)
end_ts = pd.to_datetime('2024-01-20 14:04:02', utc=True)

# Candidate services
candidates = ['carts','catalogue','orders','payment','user']

# Filter rows to window and candidate cmdb_ids
df_log_win = df_log[(df_log['_ts_dt_utc'] >= start_ts) & (df_log['_ts_dt_utc'] <= end_ts) & (df_log['cmdb_id'].isin(candidates))].copy()

# 1) Group by cmdb_id and log_name compute requested aggregates
agg_basic = df_log_win.groupby(['cmdb_id', 'log_name']).agg(
    rows_in_window=('timestamp', 'size'),
    sum_value=('value', 'sum'),
    max_value=('value', 'max')
).reset_index()

# earliest_nonzero_timestamp: first timestamp in window where value>0 (Unix seconds), None if none
nonzero = df_log_win[df_log_win['value'] > 0].groupby(['cmdb_id', 'log_name'])['timestamp'].min().rename('earliest_nonzero_timestamp').reset_index()
log_agg = agg_basic.merge(nonzero, on=['cmdb_id', 'log_name'], how='left')
# Replace missing earliest_nonzero_timestamp with pd.NA for compactness
log_agg['earliest_nonzero_timestamp'] = log_agg['earliest_nonzero_timestamp'].where(log_agg['earliest_nonzero_timestamp'].notna(), pd.NA)

# Limit to 20 rows for compactness
log_agg = log_agg.head(20)

# 2) Top 20 raw log rows where log_name=='log.error_count' and value>0, sorted by timestamp ascending
error_rows = df_log_win[(df_log_win['log_name'] == 'log.error_count') & (df_log_win['value'] > 0)].sort_values(by='timestamp', ascending=True)
error_rows_top20 = error_rows.loc[:, ['timestamp', 'cmdb_id', 'log_name', 'value']].head(20)

# Display compact results
log_agg, error_rows_top20

```
Out[4]:
```
```python
summary = (
    "Log summary for 2024-01-20 13:34:02–14:04:02 UTC (services: carts, catalogue, orders, payment, user):\n\n"
    "1) log.error_count:\n"
    "   - carts: 23 rows, sum=0, no nonzero error timestamps\n"
    "   - catalogue: 25 rows, sum=0, no nonzero error timestamps\n"
    "   - orders: 25 rows, sum=0, no nonzero error timestamps\n"
    "   - payment: 25 rows, sum=0, no nonzero error timestamps\n"
    "   - user: 25 rows, sum=0, no nonzero error timestamps\n"
    "   => Interpretation: none of the candidate services recorded any log.error_count > 0 during the incident window.\n\n"
    "2) log.row_count (per service aggregated in-window):\n"
    "   - carts: rows=23, sum=1576, max=125, earliest nonzero timestamp=1705757820\n"
    "   - catalogue: rows=25, sum=3910, max=173, earliest nonzero timestamp=1705757820\n"
    "   - orders: rows=25, sum=3076, max=259, earliest nonzero timestamp=1705757820\n"
    "   - payment: rows=25, sum=2434, max=106, earliest nonzero timestamp=1705757820\n"
    "   - user: rows=25, sum=14230, max=627, earliest nonzero timestamp=1705757820\n\n"
    "3) Raw log.error_count rows with value>0 (top 20): none (empty result).\n\n"
    "Conclusion: During the incident window, candidate services produced regular row_count logs but no recorded log.error_count > 0."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id         log_name  rows_in_window  sum_value  max_value  earliest_nonzero_timestamp
0      carts  log.error_count              23          0          0                        <NA>
1      carts    log.row_count              23       1576        125                  1705757820
2  catalogue  log.error_count              25          0          0                        <NA>
3  catalogue    log.row_count              25       3910        173                  1705757820
4     orders  log.error_count              25          0          0                        <NA>
5     orders    log.row_count              25       3076        259                  1705757820
6    payment  log.error_count              25          0          0                        <NA>
7    payment    log.row_count              25       2434        106                  1705757820
8       user  log.error_count              25          0          0                        <NA>
9       user    log.row_count              25      14230        627                  1705757820, Empty DataFrame
Columns: [timestamp, cmdb_id, log_name, value]
Index: [])```
```